# Spark Setup

In [ ]:
import glob
import os
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date, sin, cos, sqrt, atan2, radians
)
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

Accepting streams

In [ ]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")

join them with each other so easy process i guess idk ill figure out why later

In [ ]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit", "latitude", "longitude")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

In [ ]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "car_plate",
            "batch_id",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        # Append JSON Lines to a single file so batches accumulate.
        file_path = base_path
        if not file_path.endswith(".json"):
            file_path = f"{base_path}/results.json"
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        rows = batch_df.toJSON().collect()
        if not rows:
            return
        with open(file_path, "a", encoding="utf-8") as f:
            for row in rows:
                f.write(row + "\n")
    return _writer

# camera_a_instant_query = (
#     camera_a_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_a")
#     .start()
# )

# camera_b_instant_query = (
#     camera_b_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_b")
#     .start()
# )

# camera_c_instant_query = (
#     camera_c_instant_violations
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
#     .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/instant_violations_json/camera_c")
#     .start()
# )

print("Debug: Instantaneous violations have been extracted and combined.")


average speed violations time baby

In [ ]:
# A→B segment join (camera 1 to camera 2)
segment_ab = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
        
    )
)

# B→C segment join (camera 2 to camera 3)
segment_bc = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
    )
)

# segment_ab = (
#     segment_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment A→B"))
#     .start()
# )

# segment_bc = (
#     segment_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment B→C"))
#     .start()
# )

print("Debug: Segment joins have been defined for A→B and B→C.")

In [ ]:


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1_rad = radians(col(lat1))
    lon1_rad = radians(col(lon1))
    lat2_rad = radians(col(lat2))
    lon2_rad = radians(col(lon2))

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        sin(delta_lat / 2) * sin(delta_lat / 2)
        + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2) * sin(delta_lon / 2)
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance_km = R * c
    return distance_km

def compute_avg_speed(joined_segments):
    return (
        joined_segments
        .withColumn(
            "distance_km",
            calculate_haversine_distance(
                "entry_latitude", "entry_longitude",
                "exit_latitude", "exit_longitude"
            )
        )
        .withColumn(
            "travel_time_hours",
            (col("exit_time").cast("double") - col("entry_time").cast("double")) / 3600
                    )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time",
            "entry_batch_id",
            "exit_batch_id"
    )
)


average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

ab_avg_violations_query = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_ab"))
    .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/avg_violations/camera_ab")
    .start()
)

bc_avg_violations_query = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_bc"))
    .option("checkpointLocation", f"{Path('..')}/outputs/checkpoints/avg_violations/camera_bc")
    .start()
)

# avg_vio_ab_query = (
#     average_violations_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations A→B"))
#     .start()
# )

# avg_vio_bc_query = (
#     average_violations_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations B→C"))
#     .start()
# )


print("Average speed violation detection logic defined.")

## Task 2.1.3 — MongoDB Sink

Violations are written via `foreachBatch` using pymongo `bulk_write` with `UpdateOne` upserts.

**Daily merging (Task 2.1.4):** Multiple violations for the same `(car_plate, date, violation_type, camera_id)` 
are merged into one document. Each new incident is appended to an `incidents` array via `$push`, 
rather than creating a duplicate document. This matches the index defined in `mongo_setup.py` on `(car_plate, date)`.

**Idempotency:** Upserts are safe to retry — re-writing the same event just pushes a duplicate 
into the `incidents` array, which is preferable to crashing the stream.


In [ ]:
def write_violations_to_mongo(batch_df, batch_id):
    """foreachBatch sink: upsert violation records into MongoDB fit3182_a2.violations.

    Uses bulk_write for efficiency. Each violation is upserted keyed on
    (car_plate, date, violation_type, camera_id) to enforce daily merging —
    matching the compound index created in mongo_setup.py.

    Args:
        batch_df: Spark DataFrame for the current micro-batch.
        batch_id: Spark-assigned integer batch identifier (used for logging).
    """
    rows = batch_df.collect()

    if not rows:
        print(f"[Batch {batch_id}] No violations to write.")
        return

    operations = []
    for row in rows:
        # Compound upsert key — one document per car per day per camera per violation type
        filter_key = {
            "car_plate":      row["car_plate"],
            "date":           str(row["date"]),       # field name matches mongo_setup.py index
            "violation_type": row["violation_type"],
            "camera_id":      row["camera_id"],
        }

        # $setOnInsert only writes these fields when creating a new document
        # $push appends each new speed reading to the incidents array
        update_doc = {
            "$setOnInsert": {
                "speed_limit": row["speed_limit"],
                "source":      row["source"],
            },
            "$push": {
                "incidents": {
                    "speed_recorded": row["speed_recorded"],
                    "event_time":     row["event_time"],
                }
            }
        }

        # Add segment metadata for average violations only
        if row["violation_type"] == "average":
            update_doc["$setOnInsert"]["segment_start_camera"] = row["segment_start_camera"]
            update_doc["$setOnInsert"]["distance_km"] = row["distance_km"]

        operations.append(UpdateOne(filter_key, update_doc, upsert=True))

    # Open a fresh MongoClient per batch — pymongo is not serialisable across batches
    client = MongoClient(MONGO_URI)
    try:
        collection = client[MONGO_DB]["violations"]
        result = collection.bulk_write(operations, ordered=False)
        print(
            f"[Batch {batch_id}] Wrote {len(operations)} violation(s) — "
            f"upserted: {result.upserted_count}, modified: {result.modified_count}"
        )
    except Exception as exc:
        # Log and continue — do not crash the stream on a transient write error
        print(f"[Batch {batch_id}] MongoDB write error: {exc}")
    finally:
        client.close()


print("Debug: MongoDB sink function defined.")


In [ ]:
spark.streams.awaitAnyTermination()